# Chapter 24: Large Language Models. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/24-llm.R`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/24-llm.R) and [`code/24-llm.py`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/24-llm.py).

This notebook holds the exercises of the chapter that use Python; the ones that use R are in [`24-llm-R.ipynb`](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/24-llm-R.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/24-llm-Python.ipynb)

In [ ]:
# Modules that the exercises of this notebook use: numpy, scipy (all installed on Colab)

## Exercise 24.13: Retrieval Quality Metrics and Hybrid Search in RAG

Section 24.5 evaluates retrieval with Precision@k (fraction of the top-$k$ retrieved chunks that are relevant), Recall@k (fraction of all relevant chunks that appear in the top $k$), and Mean Reciprocal Rank (MRR, here the reciprocal rank $1/\text{rank}$ of the first relevant chunk). It also fuses dense and lexical search through the formula $H = (1-\alpha)\cdot\text{BM25} + \alpha\cdot\text{Semantic}$, with a typical $\alpha \approx 0.5$. This exercise works through these mechanics on a small worked example, distinct from the earlier Context Window Budget exercise, which covers token budgeting rather than retrieval quality.

A knowledge base has 6 chunks (A through F). For the query "What is the timeout setting for RFC 2616 keep-alive connections?" a retrieval system reports the scores below. Only chunks B and E actually answer the question: B states the number tersely using the exact RFC identifier and section number, while E restates the same fact in more natural, paraphrased language.

| Chunk | Semantic (cosine) | BM25 (lexical) | Relevant? |
| -- | -- | -- | -- |
| A | 0.55 | 1.0 | No |
| B | 0.42 | 9.5 | Yes |
| C | 0.20 | 0.5 | No |
| D | 0.62 | 1.5 | No |
| E | 0.81 | 3.0 | Yes |
| F | 0.15 | 2.0 | No |

a) Rank the chunks by semantic score alone. Compute Precision@3, Recall@3, and MRR.

b) Min-max normalize the semantic and BM25 columns separately to $[0,1]$, and explain why normalizing before fusing is necessary given the very different native scales of cosine similarity and BM25. Using the fusion formula with $\alpha = 0.5$ on the normalized scores, rank the chunks again and recompute Precision@3, Recall@3, and MRR.

c) Compare (a) and (b). Which relevant chunk was rescued from outside the top 3, and why? Relate this to the chapter's point that a user searching for "RFC 2616" needs lexical matching that pure semantic similarity to "HTTP specification" would miss.

d) Sweep $\alpha \in \{0, 0.3, 0.5, 0.7, 1.0\}$ and report Precision@3 and Recall@3 at each value. Is the chapter's "typical $\alpha \approx 0.5$" consistent with what you find? At what value of $\alpha$ does retrieval quality degrade, and why, given chunk B's scores?

## Exercise 24.14: Self-Consistency: Majority Voting over Reasoning Paths

Section 24.4 reports that giving PaLM 540B eight chain-of-thought (CoT) examples raised its GSM8K accuracy from 18% to 57% (Wei et al. 2023), a gain that came entirely from a better prompt. The chapter also stresses that reasoning need not be linear: Tree-of-Thought explores several branches instead of one, and Reflexion revises an answer after critiquing itself. *Self-consistency* is a third way to spend extra inference-time compute: draw several independent CoT samples for the same problem and return the most common (mode) final answer instead of trusting a single sample.

Suppose each sampled reasoning path independently reaches the correct answer with probability $p$, and otherwise lands on one of $m$ distinct wrong answers, chosen uniformly at random among the $m$ options. Majority voting returns whichever of the $m+1$ possible answers gets the most votes among $k$ independent samples, breaking ties at random.

a) Simulate this voting scheme for $p = 0.57$ (the chapter's post-CoT GSM8K accuracy) with $m=3$ wrong answers, for $k \in \{1, 3, 5, 7, 11, 21\}$ samples per problem, using at least 20,000 simulated problems per value of $k$. Report the accuracy at each $k$ and the gain of $k=11$ over the single-sample case $k=1$.

b) Consider the simpler two-outcome case: ignore how the wrong answers split, and count a vote as correct whenever more than half of the $k$ samples are correct. For odd $k$ this majority-vote probability is a binomial tail,
$$
\mathrm{P}\left(\text{majority correct}\right) = \mathrm{P}\left(X > k/2\right), \qquad X \sim \text{Binomial}(k,p), \quad \mathrm{E}\left(X\right) = kp.
$$
Compute this theoretical accuracy for $p=0.57$ at the same values of $k$ as in (a). Is it higher or lower than your simulation from (a), and why would splitting the wrong answers across $m>1$ labels make the vote easier for the correct answer to win than a strict two-outcome majority requires?

c) Repeat (a) and (b) with $p = 0.18$, the chapter's pre-CoT (direct-answer) GSM8K accuracy. What happens to voting accuracy as $k$ grows? State a general rule for when self-consistency helps versus when it hurts, and explain why the accuracy gains in (a) shrink at larger $k$ even though $p > 0.5$ throughout.

## Exercise 24.15: Tree-of-Thought and Reflexion: Search and Self-Critique

Section 24.4 presents two ways to spend extra inference-time compute beyond the self-consistency voting of Exercise 24.14: Tree-of-Thought (ToT) explores and prunes multiple partial solutions, while Reflexion revises a single line of attempts using its own written critique of what went wrong.

a) *Tree-of-Thought as pruned beam search.* Model a $d$-step puzzle (like Game of 24's three arithmetic operations) where at each step there are $b = 4$ candidate continuations, exactly one of which lies on a correct solution path. A linear CoT strategy picks one candidate uniformly at random at each of $d = 3$ steps, succeeding only if it is correct all three times. A ToT strategy scores all $b$ candidates with a noisy evaluator that ranks the true best candidate first with probability $s = 0.6$ (otherwise it orders all $b$ candidates uniformly at random), keeps the best $k = 2$, and expands both survivors; the search succeeds if the correct candidate survives the beam at every level.

- (i) Derive $p_{\text{CoT}} = (1/b)^d$ and, with per-level retention probability $r = s + (1-s)(k/b)$, $p_{\text{ToT}} = r^d$.
- (ii) Compute both.
- (iii) Compute the number of nodes each strategy evaluates: linear CoT ($d$), beam search ($b + (d-1)kb$), and exhaustive search ($b^d$). Express the beam search's savings as a multiple of the exhaustive cost.
- (iv) Confirm (ii) with a short Monte Carlo simulation, and comment on how the pattern (large accuracy gain, modest node-count increase) compares to the chapter's Game of 24 result (4% to 74%, Yao et al. (2023)).

b) *Reflexion and the independence assumption.* The chapter reports a GPT-4 agent using Reflexion raised its HumanEval success rate from $p_1 = 0.80$ to 0.91 (Shinn et al. 2023). Suppose, counterfactually, that repeated attempts at the same problem (no reflection, just resampling, checked against the same unit-test oracle) were independent Bernoulli trials each succeeding with probability $p_1$.

- (i) Compute $\mathrm{P}\left(\text{at least one success in 2 independent tries}\right)$ and compare it to the reported 0.91.
- (ii) Your answer to (i) exceeds 0.91. What does this imply about whether repeated LLM samples on the same failed problem, with no new information, can be treated as independent draws at rate $p_1$?
- (iii) Using the chapter's description of Reflexion ("the model gains short-term memory and avoids repeating past mistakes"), argue why the targeted information in a self-written critique does work that plain resampling cannot, and relate this to why the independence assumption in Exercise 24.14 (independent parallel CoT samples) is more defensible than an independence assumption across un-reflected sequential retries on the same problem.

c) Fill in a table contrasting self-consistency (Exercise 24.14), Tree-of-Thought, and Reflexion on: whether attempts are generated independently or sequentially/dependently; whether an external verifier or oracle is required; and what happens to a failed candidate.